# 25 - Context Loader Build + Entry Point Evaluation Notebook

This notebook both **implements** and **validates** the Chapter 4 entrypoint that the recommender engine will consume. It is intentionally lightweight: it focuses only on the *new* integration risks introduced in Chapter 4 and avoids duplicating the broader Chapter 3 pipeline evaluation suite.

### What this notebook builds

- **Chapter 4 context loader (wrapper)**
  - Implements `load_ch4_context()` in `src/job_intel/features/artefacts_ch4.py`.
  - The wrapper:
    - calls Chapter 3 public API (`run_positioning`) to produce `profile`, `candidates_df`, `gap_df`, and optional sensitivity outputs
    - loads aligned Chapter 3 artefacts (`jobs_df`, `skill_prob_matrix`) via `load_ch3_artefacts()`
    - loads the trained salary model (`salary_model_v4.pkl`)
    - constructs a salary prediction design matrix by combining candidate job/company codes with broadcasted user `skill_PC1..skill_PC10`

### What this notebook evaluates

- **Salary feature matrix integrity**
  - Confirms the salary design matrix contains the required columns:
    - `size_code`, `sector_code`, `state_code`, `ownership_code`, `seniority_code`, `title_rich_code`
    - `skill_PC1` … `skill_PC10`
  - Confirms row counts match the candidate set and required fields contain no missing values.

- **User PC broadcasting correctness**
  - Confirms the single user `skill_PC*` vector is correctly repeated across all candidate rows (no join/index misalignment).

- **Salary prediction smoke test**
  - Runs `salary_model.predict(...)` to confirm the end-to-end salary feature pipeline is valid and produces finite predictions.

- **Artefact alignment prerequisite (for upskilling and what-if simulation)**
  - Confirms candidate `job_id` values are aligned with the `skill_prob_matrix` universe.

### Output
- A working `load_ch4_context()` wrapper committed to `src/`.
- A compact PASS/FAIL report from `evaluation/chapter_4_entrypoint_eval.py` confirming the entrypoint is safe to use for:
  - hybrid job recommendations
  - upskilling recommendations
  - what-if simulations driven by salary predictions


## Libraries

In [18]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path
import joblib

## Path

In [19]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [20]:
from src.job_intel.positioning import run_positioning
from src.job_intel.features.artefacts_ch3 import load_ch3_artefacts
from src.job_intel.config import MODELS_DIR

def artetfacts_wrapper_ch4(skill_text = '',
                           current_state = 'ALL',
                           job_title_family = None,
                           job_title_rich = None,
                           target_sectors = None,
                           salary_target = None,
                           explain_skills = None,
                           w_skill = 0.7,
                           w_salary = 0.3,
                           top_k_gaps = 200,
                           return_top_n_jobs = 6200, # N jobs ~6100
                           run_sensitivity = False,
                           salary_model_path = MODELS_DIR / "salary_model_v4.pkl"
                           ):    
        profile, candidates_df, gap_df, sensitivity_out = run_positioning(
                                                                        skill_text = skill_text,
                                                                        current_state = current_state,
                                                                        job_title_family = job_title_family,
                                                                        job_title_rich = job_title_rich,
                                                                        target_sectors = target_sectors,
                                                                        salary_target = salary_target,
                                                                        explain_skills = explain_skills,
                                                                        w_skill = w_skill,
                                                                        w_salary = w_salary,
                                                                        top_k_gaps = top_k_gaps,
                                                                        return_top_n_jobs = return_top_n_jobs,
                                                                        run_sensitivity = run_sensitivity,
                                                                        )
        
        jobs_df, skill_prob_matrix = load_ch3_artefacts()

        salary_model = joblib.load(salary_model_path)

        # Prepare features for salary predictions
        
        model_features = candidates_df[[
                                        # Company
                                        'size_code', 'sector_code', 'state_code','ownership_code',
                                        # Role
                                        'seniority_code', 'title_rich_code'
                ]].copy() 
        
        cat_cols = ['sector_code','state_code', 'title_rich_code']
        model_features[cat_cols] = model_features[cat_cols].astype('category')

        user_pcs = profile["derived"]["skill_pcs"]

        # Guardrails
        assert len(user_pcs) == 1, f"Expected user_pcs to be 1 row, got {len(user_pcs)}"
        assert "dummy" not in model_features.columns
        assert "dummy" not in user_pcs.columns

        mf = model_features.assign(dummy=1)
        up = user_pcs.assign(dummy=1)

        user_salary_model_features = mf.merge(up, on="dummy", how="left").drop(columns="dummy")
        

        return profile, candidates_df, gap_df, sensitivity_out, jobs_df, skill_prob_matrix, salary_model, user_salary_model_features




In [21]:
profile, candidates_df, gap_df, sensitivity_out, jobs_df, skill_prob_matrix, salary_model, user_salary_model_features = artetfacts_wrapper_ch4(current_state = 'ALL',
                           job_title_family = None,
                           job_title_rich = None,
                           target_sectors = None)

In [22]:
user_salary_model_features

,size_code,sector_code,state_code,ownership_code,seniority_code,title_rich_code,skill_PC1,skill_PC2,skill_PC3,skill_PC4,skill_PC5,skill_PC6,skill_PC7,skill_PC8,skill_PC9,skill_PC10
0,0,25,17,2,7,8,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
1,6,9,11,2,2,18,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
2,0,5,1,2,7,10,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
3,7,25,15,4,7,1,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
4,1,1,2,3,9,8,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6105,2,25,15,2,9,10,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
6106,4,14,15,3,7,4,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
6107,1,14,15,3,7,4,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045
6108,4,14,15,3,7,4,-1.196572,-0.907866,-0.614978,-0.385748,-0.043992,0.176792,-0.106642,-0.077199,-0.245895,0.14045


In [23]:
# Improved version


def load_ch4_context(
    skill_text: str = "",
    current_state: str = "ALL",
    job_title_family: str | None = None,
    job_title_rich: str | None = None,
    target_sectors: list[str] | None = None,
    salary_target: float | None = None,
    explain_skills: bool | None = None,
    w_skill: float = 0.7,
    w_salary: float = 0.3,
    top_k_gaps: int = 200,
    return_top_n_jobs: int | None = 6200,  # set None for full-universe engine mode
    run_sensitivity: bool = False,
    salary_model_path=MODELS_DIR / "salary_model_v4.pkl",
) -> dict:
    """
    Chapter 4 — Context Loader / Wrapper

    Purpose
    - Provides a single, canonical entrypoint to obtain everything Chapter 4 needs:
      (1) Chapter 3 user positioning outputs (profile, candidates, gaps, sensitivity),
      (2) aligned Chapter 3 artefacts (refined jobs df + skill probability matrix),
      (3) the trained salary model artefact, and
      (4) a ready-to-predict salary feature matrix for the given user profile.

    Key behavior
    - Calls `run_positioning(...)` to build the candidate universe and user-derived features.
    - Loads persisted Chapter 3 artefacts via `load_ch3_artefacts()` (no expensive Chapter 1 reruns).
    - Broadcasts the user's skill PCs (1×10) across the candidate feature rows to form a design matrix
      suitable for salary prediction.

    Parameters
    - The parameters mirror `run_positioning(...)`. The caller should choose `return_top_n_jobs=None`
      when Chapter 4 needs a full candidate universe (recommended for recommender engine mode).

    Returns
    - dict with:
        - "profile": user profile dict including derived fields (e.g., skill PCs)
        - "candidates_df": candidate jobs after filtering + positioning indices
        - "gap_df": skill gap summary outputs (as produced by Chapter 3)
        - "sensitivity_out": sensitivity outputs (or None)
        - "jobs_df": refined, feature-complete jobs dataframe (aligned artefact)
        - "skill_prob_matrix": job×skill requirement probability matrix (aligned artefact)
        - "salary_model": loaded salary model artefact
        - "user_salary_model_features": candidate-level salary model feature matrix (with user PCs)
    """
    profile, candidates_df, gap_df, sensitivity_out = run_positioning(
        skill_text=skill_text,
        current_state=current_state,
        job_title_family=job_title_family,
        job_title_rich=job_title_rich,
        target_sectors=target_sectors,
        salary_target=salary_target,
        explain_skills=explain_skills,
        w_skill=w_skill,
        w_salary=w_salary,
        top_k_gaps=top_k_gaps,
        return_top_n_jobs=return_top_n_jobs,
        run_sensitivity=run_sensitivity,
    )

    jobs_df, skill_prob_matrix = load_ch3_artefacts()
    salary_model = joblib.load(salary_model_path)

    # Candidate-level salary features (job/company attributes).
    model_features = candidates_df[
        [
            "size_code",
            "sector_code",
            "state_code",
            "ownership_code",
            "seniority_code",
            "title_rich_code",
        ]
    ].copy()

    # If the salary model was trained with categorical handling, keep the same dtype convention.
    cat_cols = [
        "size_code",
        "sector_code",
        "state_code",
        "ownership_code",
        "seniority_code",
        "title_rich_code",
    ]
    model_features[cat_cols] = model_features[cat_cols].astype("category")

    # Broadcast user skill PCs across rows
    user_pcs = profile["derived"]["skill_pcs"]
    
    assert len(user_pcs) == 1, f"Expected user_pcs to be 1 row, got {len(user_pcs)}"
    assert "dummy" not in model_features.columns
    assert "dummy" not in user_pcs.columns

    mf = model_features.assign(dummy=1)
    up = user_pcs.assign(dummy=1)
    user_salary_model_features = mf.merge(up, on="dummy", how="left").drop(
        columns="dummy"
    )

    return {
        "profile": profile,
        "candidates_df": candidates_df,
        "gap_df": gap_df,
        "sensitivity_out": sensitivity_out,
        "jobs_df": jobs_df,
        "skill_prob_matrix": skill_prob_matrix,
        "salary_model": salary_model,
        "user_salary_model_features": user_salary_model_features,
    }


In [24]:
out = load_ch4_context(current_state = 'ALL',
                           job_title_family = None,
                           job_title_rich = None,
                           target_sectors = None)

# tests

In [ ]:
from src.job_intel.evaluation.chapter4_entrypoint_eval import evaluate_ch4_entrypoint

In [26]:
ctx = evaluate_ch4_entrypoint(current_state = 'ALL',
                           job_title_family = None,
                           job_title_rich = None,
                           target_sectors = None
)

TEST - CH4 ENTRYPOINT (MINIMAL DELTA)
------------------------------------
✅ Salary feature matrix schema + shape: PASS
✅ PC broadcasting correctness: PASS
✅ Salary predict smoke test: PASS
✅ job_id alignment to skill_prob_matrix: PASS


# == End of Notebook ==